## Load Image

In [ ]:
pip install adversarial-robustness-toolbox

In [ ]:
import os
from PIL import Image

def convert_ppm_to_png(source_directory, target_directory):
    # Create the target directory if it does not exist
    os.makedirs(target_directory, exist_ok=True)

    # Loop through all files in the source directory
    for filename in os.listdir(source_directory):
        if filename.endswith(".ppm"):
            # Construct the full file path
            file_path = os.path.join(source_directory, filename)



            # Open the .ppm image
            with Image.open(file_path) as img:
                # Convert the image to .png format
                output_file = os.path.join(target_directory, filename[:-4] + '.png')
                img.save(output_file, 'PNG')

    print("Conversion complete!")


In [ ]:
import cv2

def load_image(image_path, resize=None):
    # Use cv2 to load the image
    image = cv2.imread(image_path)

    if image is None:
        raise FileNotFoundError(f"Image file not found or unable to load: {image_path}")

    # Resize the image if resize is provided
    if resize is not None:
        image = cv2.resize(image, resize)

    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    return image

## Calculate SSIM

In [ ]:
import numpy as np
from skimage.metrics import structural_similarity as ssim
import cv2

def calculate_ssim(image1, image2):
    # Load images if paths are provided
    if isinstance(image1, str):
        image1 = cv2.imread(image1)
    if isinstance(image2, str):
        image2 = cv2.imread(image2)

    # Convert images to grayscale if they are RGB
    if len(image1.shape) > 2:
        image1 = cv2.cvtColor(image1, cv2.COLOR_BGR2GRAY)
    if len(image2.shape) > 2:
        image2 = cv2.cvtColor(image2, cv2.COLOR_BGR2GRAY)

    # Calculate the range of pixel values
    data_range = np.max(image1) - np.min(image1)

    # Calculate SSIM between the two images
    ssim_value, _ = ssim(image1, image2, full=True, data_range=data_range)

    return ssim_value*100

## Calculate Percentage Decrease

In [ ]:
def percentage_decrease(old_value, new_value):
    if old_value == 0:
        return 0.0

    if(new_value==0):
        return 100.0

    decrease = old_value - new_value
    percentage_decrease = (decrease / old_value) * 100

    return percentage_decrease

## SIFT

In [ ]:
def sift(image1_path, image2_path, threshold=0.25, ransac_threshold=3.0):
    # Load the images
    if isinstance(image1_path, str):
        img1 = cv2.imread(image1_path, cv2.IMREAD_GRAYSCALE)
    if isinstance(image2_path, str):
        img2 = cv2.imread(image2_path, cv2.IMREAD_GRAYSCALE)

    if isinstance(image1_path, np.ndarray):
        img1 = cv2.cvtColor(image1_path.astype('uint8'), cv2.COLOR_RGB2GRAY)
    if isinstance(image2_path, np.ndarray):
        img2 = cv2.cvtColor(image2_path.astype('uint8'), cv2.COLOR_RGB2GRAY)

    # Initialize SIFT detector
    sift = cv2.SIFT_create()

    # Detect and compute keypoints and descriptors
    keypoints1, descriptors1 = sift.detectAndCompute(img1, None)
    keypoints2, descriptors2 = sift.detectAndCompute(img2, None)

    # Initialize a brute-force matcher
    bf = cv2.BFMatcher()

    # Match descriptors
    matches = bf.knnMatch(descriptors1, descriptors2, k=2)

    # Apply ratio test
    good_matches = []
    for m, n in matches:
        if m.distance < threshold * n.distance:
            good_matches.append(m)

    # Convert keypoints to points
    src_pts = [keypoints1[m.queryIdx].pt for m in good_matches]
    dst_pts = [keypoints2[m.trainIdx].pt for m in good_matches]

    # Perform RANSAC for robustness
    if len(src_pts) >= 4:
        src_pts = np.float32(src_pts).reshape(-1, 1, 2)
        dst_pts = np.float32(dst_pts).reshape(-1, 1, 2)

        M, mask = cv2.findHomography(src_pts, dst_pts, cv2.RANSAC, ransac_threshold)
        mask_matches = mask.ravel().tolist()
        num_matches = sum(mask_matches)
    else:
        num_matches = 0

    return num_matches

## ORB

In [ ]:
import cv2
import numpy as np

def orb(image1_path, image2_path, threshold=0.3, ransac_threshold=3.0):
    # Load the images
    if isinstance(image1_path, str):
        img1 = cv2.imread(image1_path, cv2.IMREAD_GRAYSCALE)
    if isinstance(image2_path, str):
        img2 = cv2.imread(image2_path, cv2.IMREAD_GRAYSCALE)

    if isinstance(image1_path, np.ndarray):
        img1 = cv2.cvtColor(image1_path.astype('uint8'), cv2.COLOR_RGB2GRAY)
    if isinstance(image2_path, np.ndarray):
        img2 = cv2.cvtColor(image2_path.astype('uint8'), cv2.COLOR_RGB2GRAY)

    # Initialize ORB detector
    orb = cv2.ORB_create()

    # Detect and compute keypoints and descriptors
    keypoints1, descriptors1 = orb.detectAndCompute(img1, None)
    keypoints2, descriptors2 = orb.detectAndCompute(img2, None)

    # Initialize a brute-force matcher
    bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)

    # Match descriptors
    matches = bf.match(descriptors1, descriptors2)

    # Sort them in the order of their distance
    matches = sorted(matches, key=lambda x: x.distance)

    # Apply ratio test
    good_matches = []
    for m in matches:
        if m.distance < threshold * matches[-1].distance:
            good_matches.append(m)

    # Convert keypoints to points
    src_pts = [keypoints1[m.queryIdx].pt for m in good_matches]
    dst_pts = [keypoints2[m.trainIdx].pt for m in good_matches]

    # Perform RANSAC for robustness
    if len(src_pts) >= 4:
        src_pts = np.float32(src_pts).reshape(-1, 1, 2)
        dst_pts = np.float32(dst_pts).reshape(-1, 1, 2)

        M, mask = cv2.findHomography(src_pts, dst_pts, cv2.RANSAC, ransac_threshold)
        mask_matches = mask.ravel().tolist()
        num_matches = sum(mask_matches)
    else:
        num_matches = 0

    return num_matches


## Attacks

In [ ]:
from art.estimators.classification import BlackBoxClassifier
from art.attacks.evasion import HopSkipJump
from datetime import datetime
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.utils import to_categorical
import tensorflow as tf
import os
import shutil
import math
import gc

### SIFT

In [ ]:
data_numbers = [1,2,3,4,5,6,7,8,9,10]

def predict(x):
    gc.collect()

    outcomes = []

    x = x[0]

    noisy_matches = sift(x.astype(np.uint8), reference_image[0])
    ssim_1 = calculate_ssim(other_image[0], x.astype(np.uint8))

    if(noisy_matches<=0.1*original_matches):
        outcomes.append(1)
    else:
        outcomes.append(0)

    return to_categorical(outcomes, 2)

with tf.device('/device:GPU:0'):

    start_time = datetime.now()

    iter_step = 10


    for data_number in data_numbers:
        start_time_2 = datetime.now()
        print("*"*20)
        print(f"Attacking Dataset: {data_number}")
        print("*"*20)

        fmba = []
        fmaa = []
        per_dec = []
        ssima = []
        tt = []

        reference_image = load_image(f'dataset/{data_number}/1.png')
        x_adv = load_image('adv.jpg', (reference_image.shape[1], reference_image.shape[0]))

        classifier = BlackBoxClassifier(predict, reference_image.shape, 2, clip_values=(0, 255))

        reference_image = np.expand_dims(reference_image, axis=0)

        for j in range(2, 7):
            start_time_1 = datetime.now()

            print("*"*20)
            print(f"Attacking Image {j} of Data {data_number}")

            other_image = load_image(f"dataset/{data_number}/{j}.png")

            x_adv = load_image('adv.jpg', (other_image.shape[1], other_image.shape[0]))
            other_image = np.expand_dims(other_image, axis=0)
            x_adv = np.expand_dims(x_adv, axis=0)
            name_other_image = f'{j}.png'

            original_matches = sift(reference_image[0], other_image[0])
            fmba.append(original_matches)

            # Display images side by side with text on top
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))

            # Display first image with text
            ax1.imshow(reference_image[0].astype(np.uint8))
            ax1.set_title('Reference Image')
            ax1.axis('off')

            # Display second image with text
            ax2.imshow(other_image[0].astype(np.uint8))
            ax2.set_title(f'Image {j} of Data {data_number}')
            ax2.axis('off')

            plt.tight_layout()
            plt.show()

            print("Original Matches: ", original_matches)

            attack = HopSkipJump(classifier=classifier, targeted=True, norm=2, max_iter=0, max_eval=100, init_eval=10)

            for t in range(5):
                # Generate adversarial examples for the batch
                x_adv = attack.generate(x=other_image.astype(np.float32), y=to_categorical([1], 2), x_adv_init=x_adv.astype(np.float32), resume=True)

                print(f"SSIM of noisy image with the original: {calculate_ssim(other_image[0], x_adv[0])}")

                attack.max_iter = iter_step

            # Display images side by side with text on top
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))

            # Display first image with text
            ax1.imshow(other_image[0].astype(np.uint8))
            ax1.set_title(f'Original Image {j} of Data {data_number}')
            ax1.axis('off')

            # Display second image with text
            ax2.imshow(x_adv[0].astype(np.uint8))
            ax2.set_title(f' Noisy Image {j} of Data {data_number}')
            ax2.axis('off')

            plt.tight_layout()
            plt.show()


            print(f"Noisy Matches: {sift(x_adv[0].astype(np.uint8), reference_image[0])}")
            fmaa.append(sift(x_adv[0].astype(np.uint8), reference_image[0]))

            print(f"Percentage decrease in feature matches: {percentage_decrease(original_matches, sift(x_adv[0].astype(np.uint8), reference_image[0]))}")
            per_dec.append(percentage_decrease(original_matches, sift(x_adv[0].astype(np.uint8), reference_image[0])))

            print(f"SSIM of noisy image with the original: {calculate_ssim(other_image[0], x_adv[0])}")
            ssima.append(calculate_ssim(other_image[0], x_adv[0]))

            print(f"Time taken: {datetime.now() - start_time_1}")
            tt.append(datetime.now() - start_time_1)

            cv2.imwrite(f'dataset/{data_number}/noisy_{j}_sift.png',
                        cv2.cvtColor(x_adv[0], cv2.COLOR_BGR2RGB))

            print(f"Image {j} of Data {data_number} attack finished")
            print("*"*20)

        print("*"*20)
        print(f"Average Original Matches of Data {data_number}: {sum(fmba)/len(fmba)}")
        print(f"Average Noisy Matches of Data {data_number}: {sum(fmaa)/len(fmaa)}")
        print(f"Average Percentage Decrease in Matches of Data {data_number}: {sum(per_dec)/len(per_dec)}")
        print(f"Average SSIM of the Noisy with Original Images of Data {data_number}: {sum(ssima)/len(ssima)}")
        print("*"*20)

print(f"Total Time Taken On All Datasets: {datetime.now() - start_time}")

### ORB

In [ ]:
data_numbers = [1,2,3,4,5,6,7,8,9,10]

def predict(x):
    gc.collect()

    outcomes = []

    x = x[0]

    noisy_matches = orb(x.astype(np.uint8), reference_image[0])
    ssim_1 = calculate_ssim(other_image[0], x.astype(np.uint8))

    if(noisy_matches<=0.1*original_matches):
        outcomes.append(1)
    else:
        outcomes.append(0)

    return to_categorical(outcomes, 2)

with tf.device('/device:GPU:0'):

    start_time = datetime.now()

    iter_step = 10

    for data_number in data_numbers:
        start_time_2 = datetime.now()
        print("*"*20)
        print(f"Attacking Dataset: {data_number}")
        print("*"*20)

        fmba = []
        fmaa = []
        per_dec = []
        ssima = []
        tt = []

        reference_image = load_image(f'dataset/{data_number}/1.png')
        x_adv = load_image('adv.jpg', (reference_image.shape[1], reference_image.shape[0]))

        classifier = BlackBoxClassifier(predict, reference_image.shape, 2, clip_values=(0, 255))

        reference_image = np.expand_dims(reference_image, axis=0)

        for j in range(2, 7):
            start_time_1 = datetime.now()

            print("*"*20)
            print(f"Attacking Image {j} of Data {data_number}")

            other_image = load_image(f"dataset/{data_number}/{j}.png")

            x_adv = load_image('adv.jpg', (other_image.shape[1], other_image.shape[0]))
            other_image = np.expand_dims(other_image, axis=0)
            x_adv = np.expand_dims(x_adv, axis=0)
            name_other_image = f'{j}.png'

            original_matches = orb(reference_image[0], other_image[0])
            fmba.append(original_matches)

            # Display images side by side with text on top
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))

            # Display first image with text
            ax1.imshow(reference_image[0].astype(np.uint8))
            ax1.set_title('Reference Image')
            ax1.axis('off')

            # Display second image with text
            ax2.imshow(other_image[0].astype(np.uint8))
            ax2.set_title(f'Image {j} of Data {data_number}')
            ax2.axis('off')

            plt.tight_layout()
            plt.show()

            print("Original Matches: ", original_matches)

            attack = HopSkipJump(classifier=classifier, targeted=True, norm=2, max_iter=0, max_eval=100, init_eval=10)

            for t in range(5):
                # Generate adversarial examples for the batch
                x_adv = attack.generate(x=other_image.astype(np.float32), y=to_categorical([1], 2), x_adv_init=x_adv.astype(np.float32), resume=True)

                print(f"SSIM of noisy image with the original: {calculate_ssim(other_image[0], x_adv[0])}")

                attack.max_iter = iter_step

            # Display images side by side with text on top
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))

            # Display first image with text
            ax1.imshow(other_image[0].astype(np.uint8))
            ax1.set_title(f'Original Image {j} of Data {data_number}')
            ax1.axis('off')

            # Display second image with text
            ax2.imshow(x_adv[0].astype(np.uint8))
            ax2.set_title(f' Noisy Image {j} of Data {data_number}')
            ax2.axis('off')

            plt.tight_layout()
            plt.show()


            print(f"Noisy Matches: {orb(x_adv[0].astype(np.uint8), reference_image[0])}")
            fmaa.append(orb(x_adv[0].astype(np.uint8), reference_image[0]))

            print(f"Percentage decrease in feature matches: {percentage_decrease(original_matches, orb(x_adv[0].astype(np.uint8), reference_image[0]))}")
            per_dec.append(percentage_decrease(original_matches, orb(x_adv[0].astype(np.uint8), reference_image[0])))

            print(f"SSIM of noisy image with the original: {calculate_ssim(other_image[0], x_adv[0])}")
            ssima.append(calculate_ssim(other_image[0], x_adv[0]))

            print(f"Time taken: {datetime.now() - start_time_1}")
            tt.append(datetime.now() - start_time_1)

            cv2.imwrite(f'dataset/{data_number}/noisy_{j}_orb.png',
                        cv2.cvtColor(x_adv[0], cv2.COLOR_BGR2RGB))

            print(f"Image {j} of Data {data_number} attack finished")
            print("*"*20)

    print("*"*20)
    print(f"Average Original Matches of Data {data_number}: {sum(fmba)/len(fmba)}")
    print(f"Average Noisy Matches of Data {data_number}: {sum(fmaa)/len(fmaa)}")
    print(f"Average Percentage Decrease in Matches of Data {data_number}: {sum(per_dec)/len(per_dec)}")
    print(f"Average SSIM of the Noisy with Original Images of Data {data_number}: {sum(ssima)/len(ssima)}")
    print("*"*20)

print(f"Total Time Taken On All Datasets: {datetime.now() - start_time}")

In [ ]:
data_numbers = [1,2,3,4,5,6,7,8,9,10]

def predict(x):
    gc.collect()

    outcomes = []

    x = x[0]

    noisy_matches = orb(x.astype(np.uint8), reference_image[0])
    ssim_1 = calculate_ssim(other_image[0], x.astype(np.uint8))

    if(noisy_matches<=0.1*original_matches):
        outcomes.append(1)
    else:
        outcomes.append(0)

    return to_categorical(outcomes, 2)

with tf.device('/device:GPU:0'):

    start_time = datetime.now()

    iter_step = 10

    for data_number in data_numbers:
        start_time_2 = datetime.now()
        print("*"*20)
        print(f"Attacking Dataset: {data_number}")
        print("*"*20)

        fmba = []
        fmaa = []
        per_dec = []
        ssima = []
        tt = []

        reference_image = load_image(f'dataset/{data_number}/1.png')
        x_adv = load_image('adv.jpg', (reference_image.shape[1], reference_image.shape[0]))

        classifier = BlackBoxClassifier(predict, reference_image.shape, 2, clip_values=(0, 255))

        reference_image = np.expand_dims(reference_image, axis=0)

        for j in range(2, 7):
            start_time_1 = datetime.now()

            print("*"*20)
            print(f"Attacking Image {j} of Data {data_number}")

            other_image = load_image(f"dataset/{data_number}/{j}.png")

            x_adv = load_image('adv.jpg', (other_image.shape[1], other_image.shape[0]))
            other_image = np.expand_dims(other_image, axis=0)
            x_adv = np.expand_dims(x_adv, axis=0)
            name_other_image = f'{j}.png'

            original_matches = orb(reference_image[0], other_image[0])
            fmba.append(original_matches)

            # Display images side by side with text on top
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))

            # Display first image with text
            ax1.imshow(reference_image[0].astype(np.uint8))
            ax1.set_title('Reference Image')
            ax1.axis('off')

            # Display second image with text
            ax2.imshow(other_image[0].astype(np.uint8))
            ax2.set_title(f'Image {j} of Data {data_number}')
            ax2.axis('off')

            plt.tight_layout()
            plt.show()

            print("Original Matches: ", original_matches)

            attack = HopSkipJump(classifier=classifier, targeted=True, norm=2, max_iter=0, max_eval=100, init_eval=10)

            for t in range(5):
                # Generate adversarial examples for the batch
                x_adv = attack.generate(x=other_image.astype(np.float32), y=to_categorical([1], 2), x_adv_init=x_adv.astype(np.float32), resume=True)

                print(f"SSIM of noisy image with the original: {calculate_ssim(other_image[0], x_adv[0])}")

                attack.max_iter = iter_step

            # Display images side by side with text on top
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))

            # Display first image with text
            ax1.imshow(other_image[0].astype(np.uint8))
            ax1.set_title(f'Original Image {j} of Data {data_number}')
            ax1.axis('off')

            # Display second image with text
            ax2.imshow(x_adv[0].astype(np.uint8))
            ax2.set_title(f' Noisy Image {j} of Data {data_number}')
            ax2.axis('off')

            plt.tight_layout()
            plt.show()


            print(f"Noisy Matches: {orb(x_adv[0].astype(np.uint8), reference_image[0])}")
            fmaa.append(orb(x_adv[0].astype(np.uint8), reference_image[0]))

            print(f"Percentage decrease in feature matches: {percentage_decrease(original_matches, orb(x_adv[0].astype(np.uint8), reference_image[0]))}")
            per_dec.append(percentage_decrease(original_matches, orb(x_adv[0].astype(np.uint8), reference_image[0])))

            print(f"SSIM of noisy image with the original: {calculate_ssim(other_image[0], x_adv[0])}")
            ssima.append(calculate_ssim(other_image[0], x_adv[0]))

            print(f"Time taken: {datetime.now() - start_time_1}")
            tt.append(datetime.now() - start_time_1)

            cv2.imwrite(f'dataset/{data_number}/noisy_{j}_orb.png',
                        cv2.cvtColor(x_adv[0], cv2.COLOR_BGR2RGB))

            print(f"Image {j} of Data {data_number} attack finished")
            print("*"*20)

        print("*"*20)
        print(f"Average Original Matches of Data {data_number}: {sum(fmba)/len(fmba)}")
        print(f"Average Noisy Matches of Data {data_number}: {sum(fmaa)/len(fmaa)}")
        print(f"Average Percentage Decrease in Matches of Data {data_number}: {sum(per_dec)/len(per_dec)}")
        print(f"Average SSIM of the Noisy with Original Images of Data {data_number}: {sum(ssima)/len(ssima)}")
        print(f"Time Taken on Data {data_number}: {datetime.now()-start_time_1}")
        print("*"*20)

print(f"Total Time Taken On All Datasets: {datetime.now() - start_time}")